# Hw2.1 - Коефіцієнт народжуваності в регіонах України

In [1]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport requestsfrom io import StringIOimport warningswarnings.filterwarnings('ignore')%matplotlib inlinesns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)plt.rcParams['font.size'] = 10

## 1. Зчитування даних з Вікіпедії

In [2]:
url = 'https://uk.wikipedia.org/wiki/%D0%9D%D0%B0%D1%80%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D1%8C_%D0%B2_%D0%A3%D0%BA%D1%80%D0%B0%D1%97%D0%BD%D1%96'try:    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}    response = requests.get(url, headers=headers, timeout=10)    tables = pd.read_html(StringIO(response.text), decimal=',', thousands=' ')    df = tables[1].copy()    print('Data loaded from Wikipedia')except Exception as e:    # Fallback data    data = {        'Регіон': ['Вінницька', 'Волинська', 'Дніпропетровська', 'Донецька', 'Житомирська',                   'Закарпатська', 'Івано-Франківська', 'Київська', 'Кіровоградська', 'Луганська',                   'Львівська', 'Миколаївська', 'Одеська', 'Полтавська', 'Рівненська',                   'Сумська', 'Тернопільська', 'Харківська', 'Херсонська', 'Хмельницька',                   'Черкаська', 'Чернівецька', 'Чернігівська', 'Київ', 'Севастополь', 'Україна'],        '2010': [9.7, 11.4, 9.3, 8.5, 9.1, 11.3, 10.2, 10.3, 9.6, 8.3, 10.1, 9.0, 9.5, 9.0, 11.5, 8.8, 9.3, 9.8, 9.6, 9.2, 9.1, 10.5, 8.6, 10.5, 9.7, 9.5],        '2014': [9.2, 11.4, 9.3, None, 8.8, 11.4, 10.1, 10.7, 9.0, None, 10.6, 8.9, 9.2, 8.7, 11.6, 8.5, 9.1, 9.9, 9.0, 9.1, 9.2, 10.4, 8.3, 11.5, 9.3, 10.4],        '2019': [7.5, 9.8, 8.1, None, 7.6, 10.1, 9.2, 7.9, 7.3, None, 8.2, 7.6, 9.2, 7.8, 10.2, 7.2, 8.9, 8.7, 9.1, 9.2, 7.9, 9.8, 7.1, 9.5, 7.8, 8.6]    }    df = pd.DataFrame(data)    print('Using fallback data')print(f'Shape: {df.shape}')

Data loaded from Wikipedia
Shape: (28, 4)


In [3]:
# Обробка данихdf = df.replace('—', None)df = df.replace('-', None)# Конвертація в числовий типfor col in df.columns[1:]:    df[col] = pd.to_numeric(df[col], errors='coerce')# Видалення рядка з даними по всій країніdf = df[:-1]# Заповнення пропусків середніми значеннямиfor col in df.columns[1:]:    df[col].fillna(df[col].mean(), inplace=True)print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')print(f'Missing values: {df.isnull().sum().sum()}')

## Графік 1: Народжуваність по регіонах у 2019 році

In [4]:
col_2019 = '2019'avg_2019 = df[col_2019].mean()fig, ax = plt.subplots(figsize=(14, 10))sorted_df = df.sort_values('2019')colors = ['#e74c3c' if x > avg_2019 else '#3498db' for x in sorted_df['2019']]bars = ax.barh(sorted_df['Регіон'], sorted_df['2019'], color=colors, edgecolor='black', linewidth=0.5)ax.axvline(avg_2019, color='green', linestyle='--', linewidth=2, label=f'Середнє: {avg_2019:.2f}')ax.set_title('Коефіцієнт народжуваності по регіонах у 2019 році', fontsize=14, fontweight='bold')ax.set_xlabel('Коефіцієнт народжуваності (на 1000 населення)', fontsize=12)ax.set_ylabel('Регіон', fontsize=12)ax.legend(loc='lower right', fontsize=10)ax.grid(axis='x', alpha=0.3)plt.tight_layout()plt.show()

## Графік 2: Динаміка народжуваності за роки

In [5]:
years = ['2010', '2014', '2019']fig, ax = plt.subplots(figsize=(12, 6))for idx, row in df.iterrows():    ax.plot(years, [row['2010'], row['2014'], row['2019']], marker='o', alpha=0.5, linewidth=1)# Середні по рокахmeans = [df['2010'].mean(), df['2014'].mean(), df['2019'].mean()]ax.plot(years, means, color='red', linewidth=3, marker='s', label='Середнє по Україні', markersize=10)ax.set_title('Динаміка коефіцієнта народжуваності по роках', fontsize=14, fontweight='bold')ax.set_xlabel('Рік', fontsize=12)ax.set_ylabel('Коефіцієнт народжуваності', fontsize=12)ax.legend(loc='upper right', fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Графік 3: Розподіл народжуваності у 2019 році

In [6]:
fig, ax = plt.subplots(figsize=(10, 6))ax.hist(df['2019'], bins=10, color='#9b59b6', edgecolor='black', alpha=0.7)ax.axvline(avg_2019, color='red', linestyle='--', linewidth=2, label=f'Середнє: {avg_2019:.2f}')ax.axvline(df['2019'].median(), color='green', linestyle=':', linewidth=2, label=f'Медіана: {df["2019"].median():.2f}')ax.set_title('Розподіл коефіцієнта народжуваності у 2019 році', fontsize=14, fontweight='bold')ax.set_xlabel('Коефіцієнт народжуваності', fontsize=12)ax.set_ylabel('Кількість регіонів', fontsize=12)ax.legend(fontsize=10)ax.grid(axis='y', alpha=0.3)plt.tight_layout()plt.show()

## Графік 4: Топ-5 регіонів за народжуваністю у 2014 році

In [7]:
top_5_2014 = df.nlargest(5, '2014')[['Регіон', '2014', '2019']]fig, ax = plt.subplots(figsize=(10, 6))x = np.arange(len(top_5_2014))width = 0.35bars1 = ax.bar(x - width/2, top_5_2014['2014'], width, label='2014', color='#3498db', edgecolor='black')bars2 = ax.bar(x + width/2, top_5_2014['2019'], width, label='2019', color='#e74c3c', edgecolor='black')ax.set_xlabel('Регіон', fontsize=12)ax.set_ylabel('Коефіцієнт народжуваності', fontsize=12)ax.set_title('Топ-5 регіонів за народжуваністю у 2014 році', fontsize=14, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(top_5_2014['Регіон'], rotation=45, ha='right')ax.legend(fontsize=10)ax.grid(axis='y', alpha=0.3)# Додавання значень на стовпцяхfor bar in bars1:    height = bar.get_height()    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width() / 2, height),                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)for bar in bars2:    height = bar.get_height()    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width() / 2, height),                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)plt.tight_layout()plt.show()

## Графік 5: Теплова карта народжуваності

In [8]:
# Вибірка для теплова карти (тільки перші 15 регіонів для наочності)heatmap_data = df.head(15).set_index('Регіон')[['2010', '2014', '2019']]fig, ax = plt.subplots(figsize=(10, 8))sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Коефіцієнт'})ax.set_title('Теплова карта народжуваності по регіонах', fontsize=14, fontweight='bold')ax.set_xlabel('Рік', fontsize=12)ax.set_ylabel('Регіон', fontsize=12)plt.tight_layout()plt.show()

In [9]:
above_avg = df[df['2019'] > avg_2019]print(f'Регіони з народжуваністю вище середньої у 2019 році: {len(above_avg)}')

Регіони з народжуваністю вище середньої у 2019 році: 15


In [10]:
max_region = df.loc[df['2014'].idxmax(), 'Регіон']max_rate = df['2014'].max()print(f'Найвища народжуваність у 2014 році: {max_region} ({max_rate:.2f})')

Найвища народжуваність у 2014 році: Рівненська (11.60)
